# Reproducing the results of the manuscript

This notebook recomputes the numbers of the manuscript from the result files in `results/`, which
are the outputs of the original runs (see `results/README.md` and `docs/PROVENANCE.md`):

| Manuscript | Recomputed from |
|---|---|
| Table 1, Supplementary Table S1, Fig. 4, Supplementary Fig. S4 | per-profile test predictions of the five MethylLlama and the five MethylGPT folds and of ElasticNetCV (2,149 profiles); fold-0 validation histories |
| Fig. 4c,d and the sensitivity analyses of Results 2.4 | the same predictions with the per-profile study, individual and pretraining-exposure table |
| Fig. 2, Supplementary Fig. S1 | cross-view statistics, hard-negative retrieval and reconstruction errors of all 16,912 held-out pretraining profiles |
| Fig. 3, Supplementary Table S2, Supplementary Figs. S2 and S3 | frozen and fine-tuned CLS and mean-pooled representations of all 10,988 downstream profiles |
| Methods "Datasets", Discussion | identifier-level data-structure audit and study-level quality check |
| Supplementary Note 1 | genomic-rank pilot summaries |

No GPU, model or data download is needed; it runs in a few minutes on a laptop CPU. Every value is
compared with the value printed in the manuscript. The figures are drawn from the same files by the
scripts in `scripts/figures/`. Producing the result files themselves from the public data is described
in the README ("Reproducing the manuscript").

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import balanced_accuracy_score, r2_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder, StandardScaler

ROOT = Path.cwd().parent                     # run the notebook from tutorials/
RES = ROOT / "results"
AGE = RES / "age_benchmark"
PRE = RES / "pretrained_cls"
FT = RES / "finetuned_cls"
AUD = RES / "data_audit"

checks = []

def check(name, value, manuscript, decimals):
    # A value matches when it rounds to the number printed in the manuscript.
    ok = round(float(value), decimals) == round(float(manuscript), decimals)
    checks.append({"quantity": name, "recomputed": round(float(value), decimals + 2),
                   "manuscript": manuscript, "match": "yes" if ok else "NO"})

def report(first):
    print(pd.DataFrame(checks[first:]).to_string(index=False))

## 1. Age prediction (Table 1, Supplementary Table S1, Fig. 4)

Each ensemble averages the five fold predictions of a test profile. The MethylGPT folds were
fine-tuned from the authors' released checkpoint with their pipeline on the same profiles and
folds. ElasticNetCV is a single model; its metrics are recomputed from its per-profile test
predictions and compared with its run summary.

In [2]:
start = len(checks)
def read_folds(model):
    return [pd.read_csv(AGE / f"{model}/fold_{k}_predictions.csv", dtype={"sample_id": str}).set_index("sample_id")
            for k in range(5)]

folds, gpt_folds = read_folds("methyllama"), read_folds("methylgpt")
paired = pd.read_csv(AGE / "paired_per_subject_predictions.csv", dtype={"sample_id": str}).set_index("sample_id")
assert len(paired) == 2149 and all(len(f) == 2149 for f in folds + gpt_folds)

# the paired file holds the two five-fold ensembles
for fs, column in [(folds, "predicted_age_llama"), (gpt_folds, "predicted_age_gpt")]:
    assert np.allclose(sum(f.loc[paired.index, "predicted_age"] for f in fs) / 5, paired[column], atol=1e-9)
true = paired["true_age"].to_numpy()

def metrics(t, p):
    return np.median(np.abs(p - t)), np.mean(np.abs(p - t)), r2_score(t, p)

def fold_metrics(fs):
    return np.array([metrics(f.loc[paired.index, "true_age"].to_numpy(), f.loc[paired.index, "predicted_age"].to_numpy())
                     for f in fs])

per_fold, gpt_per_fold = fold_metrics(folds), fold_metrics(gpt_folds)
llama = metrics(true, paired["predicted_age_llama"].to_numpy())
gpt = metrics(true, paired["predicted_age_gpt"].to_numpy())
enet = json.loads((AGE / "elasticnet_results.json").read_text())["test"]
enet_pred = pd.read_csv(AGE / "elasticnet_test_predictions.csv", dtype={"sample_id": str}).set_index("sample_id").loc[paired.index]
assert np.allclose(enet_pred["true_age"], true)
enet_m = metrics(true, enet_pred["predicted_age"].to_numpy())
assert np.allclose(enet_m, (enet["medae"], enet["mae"], enet["r2"]), atol=1e-6)    # predictions = run summary

for k, paper in enumerate([(3.126, 4.439, 0.9321), (3.213, 4.458, 0.9337), (3.209, 4.570, 0.9284),
                           (3.152, 4.541, 0.9298), (3.144, 4.455, 0.9338)]):
    for name, v, p, d in zip(("MedAE", "MAE", "R2"), per_fold[k], paper, (3, 3, 4)):
        check(f"MethylLlama fold {k} {name}", v, p, d)
for name, j, p, d in [("MedAE", 0, (3.169, 0.040), 3), ("MAE", 1, (4.493, 0.059), 3), ("R2", 2, (0.9316, 0.0024), 4)]:
    check(f"MethylLlama fold mean {name}", per_fold[:, j].mean(), p[0], d)
    check(f"MethylLlama fold SD {name}", per_fold[:, j].std(ddof=1), p[1], d)
for k, paper in enumerate([(4.195, 5.855, 0.8846), (4.129, 5.891, 0.8825), (4.052, 5.705, 0.8907),
                           (4.162, 5.748, 0.8897), (3.971, 5.689, 0.8922)]):
    for name, v, p, d in zip(("MedAE", "MAE", "R2"), gpt_per_fold[k], paper, (3, 3, 4)):
        check(f"MethylGPT fold {k} {name}", v, p, d)
for name, j, p, d in [("MedAE", 0, (4.102, 0.090), 3), ("MAE", 1, (5.777, 0.091), 3), ("R2", 2, (0.8880, 0.0042), 4)]:
    check(f"MethylGPT fold mean {name}", gpt_per_fold[:, j].mean(), p[0], d)
    check(f"MethylGPT fold SD {name}", gpt_per_fold[:, j].std(ddof=1), p[1], d)
for model, m, paper in [("MethylLlama ensemble", llama, (3.112, 4.333, 0.9365)),
                        ("MethylGPT ensemble", gpt, (3.701, 5.105, 0.9110)),
                        ("ElasticNetCV", enet_m, (3.437, 4.731, 0.9322))]:
    for name, v, p, d in zip(("MedAE", "MAE", "R2"), m, paper, (3, 3, 4)):
        check(f"{model} {name}", v, p, d)
check("ElasticNetCV non-zero coefficients", json.loads((AGE / "elasticnet_results.json").read_text())["n_nonzero_coef"], 1036, 0)
for name, v, p, d in zip(("MedAE", "MAE", "R2"),
                         (enet["medae"] - llama[0], enet["mae"] - llama[1], llama[2] - enet["r2"]),
                         (0.325, 0.398, 0.0043), (3, 3, 4)):
    check(f"improvement over ElasticNetCV {name}", v, p, d)
check("test cohort: youngest age", true.min(), 0, 0)
check("test cohort: oldest age", true.max(), 103, 0)
check("test cohort: mean age", true.mean(), 45.5, 1)
for k, paper in [(5, (66.5, 61.3)), (10, (89.7, 86.4))]:
    check(f"within {k} years (%), MethylLlama", 100 * (np.abs(paired["predicted_age_llama"] - paired["true_age"]) <= k).mean(), paper[0], 1)
    check(f"within {k} years (%), MethylGPT", 100 * (np.abs(paired["predicted_age_gpt"] - paired["true_age"]) <= k).mean(), paper[1], 1)
report(start)

                           quantity  recomputed  manuscript match
           MethylLlama fold 0 MedAE    3.126300      3.1260   yes
             MethylLlama fold 0 MAE    4.439260      4.4390   yes
              MethylLlama fold 0 R2    0.932132      0.9321   yes
           MethylLlama fold 1 MedAE    3.212550      3.2130   yes
             MethylLlama fold 1 MAE    4.457620      4.4580   yes
              MethylLlama fold 1 R2    0.933656      0.9337   yes
           MethylLlama fold 2 MedAE    3.209440      3.2090   yes
             MethylLlama fold 2 MAE    4.570260      4.5700   yes
              MethylLlama fold 2 R2    0.928390      0.9284   yes
           MethylLlama fold 3 MedAE    3.151620      3.1520   yes
             MethylLlama fold 3 MAE    4.540970      4.5410   yes
              MethylLlama fold 3 R2    0.929793      0.9298   yes
           MethylLlama fold 4 MedAE    3.143780      3.1440   yes
             MethylLlama fold 4 MAE    4.454530      4.4550   yes
          

### Paired bootstrap against MethylGPT and ElasticNetCV

10,000 resamples of the 2,149 test profiles (seed 0), the same resample applied to both models.
Positive values favour MethylLlama: error differences are comparator minus MethylLlama, the $R^2$
difference is MethylLlama minus comparator. (The stored `paired_bootstrap_summary.json` and
`paired_bootstrap_vs_elasticnet_summary.json` keep the $R^2$ gap as comparator minus MethylLlama;
see `results/README.md`.) The nested age restrictions (Fig. 4c) repeat the procedure within each
subset.

In [3]:
start = len(checks)

def paired_bootstrap(t, a, b, n_boot=10000, seed=0):
    # a = MethylLlama, b = MethylGPT; returns observed differences and 95% percentile intervals
    rng = np.random.default_rng(seed)
    diff = lambda i: (lambda ma, mb: (mb[0] - ma[0], mb[1] - ma[1], ma[2] - mb[2]))(metrics(t[i], a[i]), metrics(t[i], b[i]))
    obs = diff(np.arange(len(t)))
    boot = np.array([diff(rng.integers(0, len(t), size=len(t))) for _ in range(n_boot)])
    return obs, np.percentile(boot, [2.5, 97.5], axis=0).T

a, b = paired["predicted_age_llama"].to_numpy(), paired["predicted_age_gpt"].to_numpy()
obs, ci = paired_bootstrap(true, a, b)
for j, (name, d, paper) in enumerate([("MedAE", 3, (0.589, 0.338, 0.815)), ("MAE", 3, (0.773, 0.583, 0.968)),
                                      ("R2", 4, (0.0254, 0.0183, 0.0329))]):
    check(f"paired improvement {name}", obs[j], paper[0], d)
    check(f"  95% CI low", ci[j, 0], paper[1], d)
    check(f"  95% CI high", ci[j, 1], paper[2], d)

en = enet_pred["predicted_age"].to_numpy()
obs_e, ci_e = paired_bootstrap(true, a, en)
for j, (name, d, paper) in enumerate([("MedAE", 3, (0.325, 0.072, 0.594)), ("MAE", 3, (0.398, 0.197, 0.609)),
                                      ("R2", 4, (0.0043, -0.0029, 0.0113))]):
    check(f"paired improvement over ElasticNetCV {name}", obs_e[j], paper[0], d)
    check(f"  95% CI low", ci_e[j, 0], paper[1], d)
    check(f"  95% CI high", ci_e[j, 1], paper[2], d)

stored = json.loads((AGE / "age_restricted_paired_bootstrap_summary.json").read_text())["subsets"]
for min_age, paper in [(18, (1776, 0.680, 0.417, 0.948)), (40, (1449, 0.636, 0.265, 0.920))]:
    s = true >= min_age
    obs_s, ci_s = paired_bootstrap(true[s], a[s], b[s])
    check(f"age >= {min_age}: n", s.sum(), paper[0], 0)
    check(f"age >= {min_age}: paired MedAE improvement", obs_s[0], paper[1], 3)
    check(f"  95% CI low", ci_s[0, 0], paper[2], 3)
    check(f"  95% CI high", ci_s[0, 1], paper[3], 3)
    key = "adults_age_ge_18" if min_age == 18 else "older_age_ge_40"
    assert np.allclose([obs_s[0], *ci_s[0]], [stored[key]["delta_medae"], *stored[key]["delta_medae_ci_95"]], atol=1e-6)
report(start)

                                  quantity  recomputed  manuscript match
                  paired improvement MedAE    0.588520      0.5890   yes
                                95% CI low    0.337980      0.3380   yes
                               95% CI high    0.814890      0.8150   yes
                    paired improvement MAE    0.772520      0.7730   yes
                                95% CI low    0.583100      0.5830   yes
                               95% CI high    0.968250      0.9680   yes
                     paired improvement R2    0.025445      0.0254   yes
                                95% CI low    0.018342      0.0183   yes
                               95% CI high    0.032950      0.0329   yes
paired improvement over ElasticNetCV MedAE    0.324560      0.3250   yes
                                95% CI low    0.072050      0.0720   yes
                               95% CI high    0.594100      0.5940   yes
  paired improvement over ElasticNetCV MAE    0.397

### Sensitivity of the paired comparison (Fig. 4c,d)

`results/data_audit/downstream_exposure_per_profile.csv` gives, for every downstream profile, its study,
the individual identifier where public metadata provide one, and its pretraining-exposure class
(`scripts/data/data_structure_audit.py`). Identifier keys shared by more than 20 profiles are labels,
not people, and are ignored. Three analyses: removing the test profiles whose individual also occurs
in development (profile bootstrap); resampling whole studies (cluster bootstrap); the subset of test
profiles with no profile of the same individual in the pretraining training partition (classes 4–6).

In [4]:
start = len(checks)
ex = pd.read_csv(AUD / "downstream_exposure_per_profile.csv", dtype={"sample_id": str})
bench = ex[ex["bench"] != "excluded"]
size = bench.groupby("individual").size()
person = bench["individual"].where(~bench["individual"].isin(size[size > 20].index))
dev_people = set(person[bench["bench"] == "development"].dropna())
tx = ex.set_index("sample_id").loc[paired.index]
ind = tx["individual"].where(tx["individual"].isin(set(person.dropna())))
shared = ind.isin(dev_people).to_numpy()
check("test profiles with an individual identifier", ind.notna().sum(), 549, 0)
check("  sharing the individual with development", shared.sum(), 33, 0)

keep = ~shared
obs_k, ci_k = paired_bootstrap(true[keep], a[keep], b[keep])
check("shared individuals removed: n", keep.sum(), 2116, 0)
for j, (name, d, paper) in enumerate([("MedAE", 3, (0.568, 0.299, 0.784)), ("MAE", 3, (0.741, 0.547, 0.936)),
                                      ("R2", 4, (0.0250, 0.0177, 0.0327))]):
    check(f"  paired improvement {name}", obs_k[j], paper[0], d)
    check(f"  95% CI low", ci_k[j, 0], paper[1], d)
    check(f"  95% CI high", ci_k[j, 1], paper[2], d)

study = tx["study"].to_numpy()
groups = [np.where(study == s)[0] for s in sorted(set(study))]
rng = np.random.default_rng(0)
diff = lambda i: (lambda ma, mb: (mb[0] - ma[0], mb[1] - ma[1], ma[2] - mb[2]))(metrics(true[i], a[i]), metrics(true[i], b[i]))
boot = np.array([diff(np.concatenate([groups[j] for j in rng.integers(0, len(groups), len(groups))])) for _ in range(10000)])
ci_c = np.percentile(boot, [2.5, 97.5], axis=0).T
check("studies in the test set", len(groups), 77, 0)
for j, (name, d, paper) in enumerate([("MedAE", 3, (0.589, 0.302, 0.850)), ("MAE", 3, (0.773, 0.526, 1.146)),
                                      ("R2", 4, (0.0254, 0.0142, 0.0456))]):
    check(f"study-resampled {name}: 95% CI low", ci_c[j, 0], paper[1], d)
    check(f"  95% CI high", ci_c[j, 1], paper[2], d)

err_l, err_g = np.abs(a - true), np.abs(b - true)
per = pd.DataFrame({"study": study, "l": err_l, "g": err_g}).groupby("study").agg(n=("l", "size"), l=("l", "mean"), g=("g", "mean"))
check("studies with lower MAE for MethylLlama", (per.l < per.g).sum(), 61, 0)
check("studies with >= 10 test profiles", (per.n >= 10).sum(), 50, 0)
check("  of which lower MAE for MethylLlama", ((per.l < per.g) & (per.n >= 10)).sum(), 41, 0)

cls_ = tx["exposure"].str[0].astype(int).to_numpy()
s = cls_ >= 4
obs_x, ci_x = paired_bootstrap(true[s], a[s], b[s])
check("no profile of the individual in pretraining training: n", s.sum(), 626, 0)
for j, (name, d, paper) in enumerate([("MedAE", 3, (0.396, -0.086, 0.886)), ("MAE", 3, (0.744, 0.372, 1.112)),
                                      ("R2", 4, (0.0249, 0.0098, 0.0395))]):
    check(f"  paired improvement {name}", obs_x[j], paper[0], d)
    check(f"  95% CI low", ci_x[j, 0], paper[1], d)
    check(f"  95% CI high", ci_x[j, 1], paper[2], d)

g = json.loads((AUD / "paired_bootstrap_grouped_sensitivity.json").read_text())
assert np.isclose(g["A_donor_disjoint_test_set"]["medae"]["ci_95"][0], ci_k[0, 0]) and np.isclose(g["B_study_cluster_bootstrap"]["mae"]["ci_95"][1], ci_c[1, 1])
report(start)

                                               quantity  recomputed  manuscript match
            test profiles with an individual identifier  549.000000    549.0000   yes
                sharing the individual with development   33.000000     33.0000   yes
                          shared individuals removed: n 2116.000000   2116.0000   yes
                               paired improvement MedAE    0.567680      0.5680   yes
                                             95% CI low    0.299480      0.2990   yes
                                            95% CI high    0.784330      0.7840   yes
                                 paired improvement MAE    0.740940      0.7410   yes
                                             95% CI low    0.546850      0.5470   yes
                                            95% CI high    0.936160      0.9360   yes
                                  paired improvement R2    0.024981      0.0250   yes
                                             95% CI lo

### Fold 0: pretrained versus random initialisation (Supplementary Fig. S4b)

Validation trajectories of the two fold-0 runs, read from their training logs.

In [5]:
start = len(checks)
pre = pd.read_csv(AGE / "fold0_train_history.csv").set_index("epoch")
rnd = pd.read_csv(AGE / "random_init_fold0_history.csv").set_index("epoch")
check("epoch 148 validation MedAE, pretrained", pre.loc[148, "val/medae"], 2.875, 3)
check("epoch 148 validation MedAE, random init", rnd.loc[148, "val/medae"], 3.969, 3)
check("epoch 148 validation R2, pretrained", pre.loc[148, "val/r2"], 0.931, 3)
check("epoch 148 validation R2, random init", rnd.loc[148, "val/r2"], 0.854, 3)
check("best validation MedAE, pretrained", pre["val/medae"].min(), 2.625, 3)
check("  at epoch", pre["val/medae"].idxmin(), 92, 0)
check("best validation MedAE, random init", rnd["val/medae"].min(), 3.804, 3)
check("  at epoch", rnd["val/medae"].idxmin(), 187, 0)
check("last recorded epoch, pretrained", pre.index.max(), 164, 0)
check("last recorded epoch, random init", rnd.index.max(), 187, 0)
report(start)

                               quantity  recomputed  manuscript match
 epoch 148 validation MedAE, pretrained     2.87500       2.875   yes
epoch 148 validation MedAE, random init     3.96875       3.969   yes
    epoch 148 validation R2, pretrained     0.93119       0.931   yes
   epoch 148 validation R2, random init     0.85407       0.854   yes
      best validation MedAE, pretrained     2.62500       2.625   yes
                               at epoch    92.00000      92.000   yes
     best validation MedAE, random init     3.80428       3.804   yes
                               at epoch   187.00000     187.000   yes
        last recorded epoch, pretrained   164.00000     164.000   yes
       last recorded epoch, random init   187.00000     187.000   yes


## 2. Cross-view consistency, hard negatives and reconstruction (Fig. 2, Supplementary Fig. S1)

For every draw, the per-profile arrays hold the cosine similarity of each profile's two views and
the rank of its partner among all candidates (rank 0 = nearest neighbour). Top-1 retrieval is the
fraction of rank 0. The unmatched similarity is the mean over all $16{,}912 \times 16{,}911$
non-partner pairs, which were not stored individually; it is read from the run records.

In [6]:
start = len(checks)
VC = PRE / "view_consistency_heldout_full/per_run"
runs = {}
for f in sorted(VC.glob("*_seed*.json")):
    j, z = json.loads(f.read_text()), np.load(f.with_suffix(".npz"))
    runs.setdefault(j["condition"], []).append({
        "matched": z["all_matched_cos"].astype(np.float64).mean(),
        "unmatched": j["pools"]["all"]["unmatched_cos"],
        "top1": 100 * (z["all_rank"] < 1).mean(),
        "top1_full": 100 * (z["fully_measured_rank"] < 1).mean(),
        "n": len(z["all_rank"]), "n_full": len(z["fully_measured_rank"])})
r = {c: pd.DataFrame(v) for c, v in runs.items()}

check("held-out profiles", r["overlap"]["n"].iloc[0], 16912, 0)
check("chance top-1 (%)", 100 / r["overlap"]["n"].iloc[0], 0.0059, 4)
check("overlap: matched cosine, mean", r["overlap"]["matched"].mean(), 0.99549, 5)
check("overlap: matched cosine, SD", r["overlap"]["matched"].std(), 0.00001, 5)
check("overlap: unmatched cosine, mean", r["overlap"]["unmatched"].mean(), 0.30952, 5)
check("overlap: top-1 (%)", r["overlap"]["top1"].mean(), 98.06, 2)
check("  lowest draw", r["overlap"]["top1"].min(), 98.05, 2)
check("  highest draw", r["overlap"]["top1"].max(), 98.07, 2)
check("disjoint: matched cosine, mean", r["disjoint"]["matched"].mean(), 0.99139, 5)
check("disjoint: matched cosine, SD", r["disjoint"]["matched"].std(), 0.00002, 5)
check("disjoint: top-1 (%)", r["disjoint"]["top1"].mean(), 89.38, 2)
check("  lowest draw", r["disjoint"]["top1"].min(), 89.29, 2)
check("  highest draw", r["disjoint"]["top1"].max(), 89.45, 2)
check("pattern-only: top-1 (%)", r["pattern_only"]["top1"].iloc[0], 0.438, 3)
check("profiles >= 99% measured", r["pattern_only"]["n_full"].iloc[0], 7845, 0)
check("  pattern-only top-1 (%)", r["pattern_only"]["top1_full"].iloc[0], 0.025, 3)
check("  chance top-1 (%)", 100 / r["pattern_only"]["n_full"].iloc[0], 0.0127, 4)
report(start)

                       quantity   recomputed  manuscript match
              held-out profiles 16912.000000 16912.00000   yes
               chance top-1 (%)     0.005913     0.00590   yes
  overlap: matched cosine, mean     0.995494     0.99549   yes
    overlap: matched cosine, SD     0.000008     0.00001   yes
overlap: unmatched cosine, mean     0.309523     0.30952   yes
             overlap: top-1 (%)    98.058600    98.06000   yes
                    lowest draw    98.048700    98.05000   yes
                   highest draw    98.066500    98.07000   yes
 disjoint: matched cosine, mean     0.991395     0.99139   yes
   disjoint: matched cosine, SD     0.000021     0.00002   yes
            disjoint: top-1 (%)    89.376400    89.38000   yes
                    lowest draw    89.285700    89.29000   yes
                   highest draw    89.445400    89.45000   yes
        pattern-only: top-1 (%)     0.437560     0.43800   yes
       profiles >= 99% measured  7845.000000  7845.0000

### Retrieval against hard negatives (Fig. 2a,b)

Per query, the matched view competes with 31 negative views drawn from pools of increasing
similarity (chance 1/32 = 3.1%); five negative draws per pool and view draw, duplicates and known
same-individual profiles never used as negatives (`scripts/evaluation/hard_negative_heldout.py`).
Top-1 rates are read from the run summary and averaged over the three view draws; the margins of
Fig. 2a are recomputed from the per-query file (disjoint views, first view and negative draw).

In [7]:
start = len(checks)
HN = PRE / "hard_negative_heldout"
hs = json.loads((HN / "summary.json").read_text())
pq = np.load(HN / "per_query.npz")
top1 = lambda view, pool: 100 * np.mean([hs["conditions"][f"{view}_seed{k}"][pool]["top1"] for k in range(3)])
pools = ["random", "same tissue", "same study", "same study+tissue", "same study+tissue+sex"]
check("chance with 31 negatives (%)", 100 / 32, 3.1, 1)
for pool, n in zip(pools, (12924, 12818, 4891, 4536, 2088)):
    check(f"queries, pool '{pool}'", hs["conditions"]["disjoint_seed0"][pool]["n_queries"], n, 0)
check("overlapping views: top-1 above 98% in every pool", min(top1("overlap", p) for p in pools + ["K nearest neighbours of target"]) > 98, True, 0)
check("disjoint views, random negatives: top-1 above 99.9%", top1("disjoint", "random") > 99.9, True, 0)
check("disjoint views, same study (%)", top1("disjoint", "same study"), 94.3, 1)
check("disjoint views, same study + tissue + sex (%)", top1("disjoint", "same study+tissue+sex"), 90.6, 1)
pos = pq["disjoint_seed0__pos"]
for pool, paper in [("random", 0.25), ("same study+tissue+sex", 0.01)]:
    h = pq[f"disjoint_seed0__{pool}"]; ok = ~np.isnan(h)
    check(f"median margin, disjoint views, pool '{pool}'", np.median(pos[ok] - h[ok]), paper, 2)
po_study = 100 * hs["conditions"]["pattern_only_seed0"]["same study"]["top1"]
print(f"pattern-only control against same-study negatives: {po_study:.1f}% (chance 3.1%; 'close to chance' in the text)")
report(start)

pattern-only control against same-study negatives: 4.6% (chance 3.1%; 'close to chance' in the text)
                                                   quantity  recomputed manuscript match
                               chance with 31 negatives (%)      3.1250        3.1   yes
                                     queries, pool 'random'  12924.0000      12924   yes
                                queries, pool 'same tissue'  12818.0000      12818   yes
                                 queries, pool 'same study'   4891.0000       4891   yes
                          queries, pool 'same study+tissue'   4536.0000       4536   yes
                      queries, pool 'same study+tissue+sex'   2088.0000       2088   yes
           overlapping views: top-1 above 98% in every pool      1.0000       True   yes
        disjoint views, random negatives: top-1 above 99.9%      1.0000       True   yes
                             disjoint views, same study (%)     94.3040       94.3   yes
         

### Reconstruction of withheld CpGs (Fig. 2c, Supplementary Fig. S1c)

Per-profile MSE over the measured CpGs withheld from the encoder input, for the model and three
controls under identical masks, on all 16,912 held-out profiles. Pearson r was accumulated over all
withheld positions during the run and is read from its summary; the stored 2-D histogram of observed
against predicted values (Supplementary Fig. S1c) must contain every one of those positions. Bootstrap
intervals are for the paired per-profile control-minus-model difference (10,000 resamples, seed 0).

In [8]:
start = len(checks)
R = PRE / "reconstruction_withheld"
rs = json.loads((R / "reconstruction_withheld_summary.json").read_text())
pm = pd.read_csv(R / "per_sample_metrics.csv")
hist = np.load(R / "obs_pred_hist2d.npz")

check("profiles", len(pm), 16912, 0)
check("withheld positions (x 1e8)", rs["n_withheld_positions_total"] / 1e8, 3.80, 2)
check("  = per-profile counts", pm["n_withheld"].sum() == rs["n_withheld_positions_total"], True, 0)
check("  = 2-D histogram total", int(hist["counts"].sum()) == rs["n_withheld_positions_total"], True, 0)
check("Pearson r, all withheld positions", rs["pearson_withheld_all_positions"], 0.973, 3)
check("Pearson r, CpG-centred", rs["pearson_withheld_cpg_centred_all_positions"], 0.908, 3)
for col, name, paper in [("model_mse", "model", 0.0055), ("b_mean_mse", "per-CpG mean", 0.0307),
                         ("b_random_mse", "random vectors", 0.0507), ("b_shuffle_mse", "shuffled CLS", 0.0553)]:
    check(f"mean MSE, {name}", pm[col].mean(), paper, 4)

rng = np.random.default_rng(0)
stored = json.loads((R / "paired_bootstrap_ci.json").read_text())
for col, name in (("b_mean_mse", "Per-CpG mean"), ("b_shuffle_mse", "Shuffled CLS"), ("b_random_mse", "Random CLS")):
    d = (pm[col] - pm["model_mse"]).to_numpy()
    lo, hi = np.percentile([d[rng.integers(0, len(d), len(d))].mean() for _ in range(10000)], [2.5, 97.5])
    print(f"{name:13s} - model: {d.mean():.5f}  95% CI {lo:.5f} to {hi:.5f}")
    assert [round(lo, 5), round(hi, 5)] == stored[name]["ci95"]
    check(f"CI of {name} - model above zero", lo > 0, True, 0)
report(start)

Per-CpG mean  - model: 0.02514  95% CI 0.02484 to 0.02543


Shuffled CLS  - model: 0.04978  95% CI 0.04927 to 0.05028


Random CLS    - model: 0.04514  95% CI 0.04482 to 0.04547
                             quantity   recomputed manuscript match
                             profiles 16912.000000      16912   yes
           withheld positions (x 1e8)     3.803300        3.8   yes
                 = per-profile counts     1.000000       True   yes
                = 2-D histogram total     1.000000       True   yes
    Pearson r, all withheld positions     0.972800      0.973   yes
               Pearson r, CpG-centred     0.907590      0.908   yes
                      mean MSE, model     0.005513     0.0055   yes
               mean MSE, per-CpG mean     0.030650     0.0307   yes
             mean MSE, random vectors     0.050656     0.0507   yes
               mean MSE, shuffled CLS     0.055289     0.0553   yes
CI of Per-CpG mean - model above zero     1.000000       True   yes
CI of Shuffled CLS - model above zero     1.000000       True   yes
  CI of Random CLS - model above zero     1.000000       T

## 3. Frozen representations (Fig. 3, Supplementary Table S2)

Probes use the original partition of the downstream collection: fitted on the training and
validation profiles, evaluated on the test profiles, features standardised on the fitted profiles,
ridge $\alpha = 1$. The classification probes keep classes with at least 30 profiles.

In [9]:
start = len(checks)
meta = pd.read_csv(PRE / "metadata.csv")
cls = np.load(PRE / "embeddings_cls.npy").astype(np.float64)
mean_pooled = np.load(PRE / "embeddings_mean.npy").astype(np.float64)
age, split = meta["age"].to_numpy(float), meta["split"].to_numpy()
fit, ev = split != "test", split == "test"
check("profiles", len(meta), 10988, 0)

def ridge(X, tr, te):
    s = StandardScaler().fit(X[tr])
    p = Ridge(alpha=1.0).fit(s.transform(X[tr]), age[tr]).predict(s.transform(X[te]))
    return r2_score(age[te], p), np.median(np.abs(p - age[te]))

r2, medae = ridge(cls, fit, ev)
check("age probe, CLS: R2", r2, 0.866, 3)
check("age probe, CLS: MedAE", medae, 5.18, 2)
r2, medae = ridge(mean_pooled, fit, ev)
check("age probe, mean-pooled: R2", r2, 0.795, 3)
check("age probe, mean-pooled: MedAE", medae, 6.53, 2)
check("probe fitting profiles", fit.sum(), 8724, 0)
check("probe evaluation profiles", ev.sum(), 2264, 0)

def classify(column):
    labels = meta[column].astype(str).to_numpy()
    counts = pd.Series(labels).value_counts()
    keep = np.isin(labels, counts[counts >= 30].index) & (labels != "nan")
    tr, te = keep & fit, keep & ev
    enc = LabelEncoder().fit(labels[keep])
    s = StandardScaler().fit(cls[tr])
    model = LogisticRegression(max_iter=2000).fit(s.transform(cls[tr]), enc.transform(labels[tr]))
    acc = balanced_accuracy_score(enc.transform(labels[te]), model.predict(s.transform(cls[te])))
    return acc, len(enc.classes_), te.sum()

for column, paper in [("tissue_type", (0.936, 37, 2197, 0.027)), ("dataset", (0.956, 85, 2006, 0.012))]:
    acc, k, n = classify(column)
    check(f"{column} probe: balanced accuracy", acc, paper[0], 3)
    check(f"  classes", k, paper[1], 0)
    check(f"  evaluation profiles", n, paper[2], 0)
    check(f"  chance", 1 / k, paper[3], 3)

unit = cls / (np.linalg.norm(cls, axis=1, keepdims=True) + 1e-9)
nn = NearestNeighbors(n_neighbors=11).fit(unit).kneighbors(unit)[1][:, 1:]
tissue, dataset = meta["tissue_type"].astype(str).to_numpy(), meta["dataset"].astype(str).to_numpy()
same_t, same_d = tissue[nn] == tissue[:, None], dataset[nn] == dataset[:, None]
check("10-NN sharing tissue and dataset (%)", 100 * (same_t & same_d).mean(), 84.6, 1)
check("10-NN sharing tissue only (%)", 100 * (same_t & ~same_d).mean(), 7.5, 1)
check("10-NN sharing dataset only (%)", 100 * (same_d & ~same_t).mean(), 1.5, 1)

def geometry(X):
    s = np.linalg.svd(X - X.mean(0), compute_uv=False)
    p = s**2 / (s**2).sum()
    unit = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    i = np.random.default_rng(0).choice(len(X), 4000, replace=False)
    g = unit[i] @ unit[i].T
    return np.exp(-(p * np.log(p + 1e-12)).sum()), g[np.triu_indices(4000, 1)].mean(), np.linalg.norm(X, axis=1).mean()

for name, X, paper in [("CLS", cls, (18.12, 0.500, 10.01)), ("mean-pooled", mean_pooled, (3.40, 0.867, 5.36))]:
    rank, aniso, norm = geometry(X)
    check(f"{name}: effective rank", rank, paper[0], 2)
    check(f"{name}: mean cosine anisotropy", aniso, paper[1], 3)
    check(f"{name}: mean L2 norm", norm, paper[2], 2)
cis = json.loads((PRE / "probe_cis.json").read_text())        # Fig. 3c,d: estimates and 95% bootstrap CIs
for key, value, paper in [("age_cls", "r2", 0.866), ("age_mean", "r2", 0.795), ("tissue_type", "balanced_acc", 0.936), ("dataset", "balanced_acc", 0.956)]:
    check(f"probe_cis.json {key}", cis[key][value], paper, 3)
    assert cis[key]["ci95"][0] < cis[key][value] < cis[key]["ci95"][1]
report(start)

/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


                            quantity  recomputed  manuscript match
                            profiles 10988.00000   10988.000   yes
                  age probe, CLS: R2     0.86632       0.866   yes
               age probe, CLS: MedAE     5.17560       5.180   yes
          age probe, mean-pooled: R2     0.79527       0.795   yes
       age probe, mean-pooled: MedAE     6.53370       6.530   yes
              probe fitting profiles  8724.00000    8724.000   yes
           probe evaluation profiles  2264.00000    2264.000   yes
tissue_type probe: balanced accuracy     0.93579       0.936   yes
                             classes    37.00000      37.000   yes
                 evaluation profiles  2197.00000    2197.000   yes
                              chance     0.02703       0.027   yes
    dataset probe: balanced accuracy     0.95582       0.956   yes
                             classes    85.00000      85.000   yes
                 evaluation profiles  2006.00000    2006.000  

### Study structure within tissues (Results 2.3)

For whole blood, cord blood and placenta (the tissues with at least two studies of at least 100
profiles; GSE56105 excluded), the fraction of the ten nearest same-tissue neighbours from the
profile's own study, with its chance level, overall and with neighbours restricted to the same sex
and age decade.

In [10]:
start = len(checks)
ok = (meta["dataset"] != "E-GEOD-56105").to_numpy()
wt = {}
for t, g in meta[ok].groupby("tissue_type"):
    vc = g["dataset"].value_counts()
    if (vc >= 100).sum() < 2:
        continue
    i = g.index.to_numpy(); ds = g["dataset"].astype(str).to_numpy()
    S = unit[i] @ unit[i].T; np.fill_diagonal(S, -np.inf)
    nn10 = np.argsort(-S, axis=1)[:, :10]
    p = vc / vc.sum()
    sex, dec = g["gender"].astype(str).to_numpy(), np.floor(g["age"].to_numpy(float) / 10)
    cell = np.array([f"{a_}|{b_}" for a_, b_ in zip(sex, dec)]); valid = (sex != "nan") & ~np.isnan(dec)
    hits, exp_ = [], []
    for q in range(len(g)):
        if not valid[q]:
            continue
        cand = np.where(valid & (cell == cell[q]))[0]; cand = cand[cand != q]
        if len(cand) < 10:
            continue
        top = cand[np.argsort(-S[q, cand])[:10]]
        hits.append((ds[top] == ds[q]).mean()); exp_.append((ds[cand] == ds[q]).mean())
    wt[t] = (100 * (ds[nn10] == ds[:, None]).mean(), 100 * float(((vc - 1) / (len(g) - 1) * p).sum()), 100 * np.mean(hits), 100 * np.mean(exp_))
wt = pd.DataFrame(wt, index=["same study", "chance", "matched same study", "matched chance"]).T
print(wt.round(1).to_string(), "\n")
check("tissues", len(wt), 3, 0)
for col, lo_, hi_ in [("same study", 85, 86), ("chance", 8, 21), ("matched same study", 79, 86), ("matched chance", 23, 39)]:
    check(f"{col} (%), lowest tissue", wt[col].min(), lo_, 0)
    check(f"{col} (%), highest tissue", wt[col].max(), hi_, 0)
stored = json.loads((PRE / "within_tissue_study_cls.json").read_text())["tissues"]
assert all(np.isclose(100 * stored[t]["same_study_fraction"], wt.loc[t, "same study"]) for t in wt.index)
report(start)

             same study  chance  matched same study  matched chance
blood cord         85.3    15.5                82.3            29.1
blood whole        85.2     7.7                78.7            23.0
placenta           86.2    20.6                86.2            39.2 

                              quantity  recomputed  manuscript match
                               tissues        3.00           3   yes
         same study (%), lowest tissue       85.17          85   yes
        same study (%), highest tissue       86.23          86   yes
             chance (%), lowest tissue        7.72           8   yes
            chance (%), highest tissue       20.56          21   yes
 matched same study (%), lowest tissue       78.70          79   yes
matched same study (%), highest tissue       86.21          86   yes
     matched chance (%), lowest tissue       22.97          23   yes
    matched chance (%), highest tissue       39.20          39   yes


### Study-held-out age probe (Fig. 3e)

GSE56105 (E-GEOD-56105) is excluded after the study-level quality check (section 5). For every other
study with at least 100 profiles, the ridge probe is fitted on all other profiles and evaluated on
that study. The summary uses the studies with an age SD of at least 10 years. The in-distribution
reference is a random 80/20 split of the same profiles (seed 0).

In [11]:
start = len(checks)
qc = dataset != "E-GEOD-56105"
idx_qc = np.where(qc)[0]
rng = np.random.default_rng(0)
perm = idx_qc[rng.permutation(len(idx_qc))]; cut = int(0.8 * len(idx_qc))
tr, te = np.zeros(len(cls), bool), np.zeros(len(cls), bool)
tr[perm[:cut]], te[perm[cut:]] = True, True
r2, medae = ridge(cls, tr, te)
check("in-distribution 80/20: R2", r2, 0.877, 3)
check("in-distribution 80/20: MedAE", medae, 4.65, 2)
check("in-distribution 80/20: evaluation profiles", te.sum(), 2142, 0)

rows = []
counts = pd.Series(dataset[qc]).value_counts()
for study in counts[counts >= 100].index:
    held = dataset == study
    r2, medae = ridge(cls, qc & ~held, held)
    rows.append({"study": study, "n": held.sum(), "age_sd": age[held].std(), "R2": r2, "MedAE": medae})
lodo = pd.DataFrame(rows)
wide = lodo[lodo["age_sd"] >= 10].sort_values("R2", ascending=False)
print(wide.round(3).to_string(index=False), "\n")

rho_sd, p_sd = spearmanr(wide["R2"], wide["age_sd"])
rho_n, p_n = spearmanr(wide["R2"], wide["n"])
check("eligible studies", len(wide), 10, 0)
check("median study-held-out R2", wide["R2"].median(), 0.294, 3)
check("median study-held-out MedAE", wide["MedAE"].median(), 7.56, 2)
check("studies with negative R2", (wide["R2"] < 0).sum(), 2, 0)
check("Spearman rho, R2 vs age SD", rho_sd, 0.32, 2)
check("  P", p_sd, 0.37, 2)
check("Spearman rho, R2 vs study size", rho_n, 0.30, 2)
check("  P", p_n, 0.40, 2)
shipped = pd.read_csv(PRE / "lodo_age_probe.csv")
shipped = shipped[(shipped["age_std"] >= 10) & shipped["R2_LODO"].notna()].set_index("dataset")
assert set(shipped.index) == set(wide["study"])
assert np.allclose(shipped.loc[wide["study"], "R2_LODO"], wide["R2"], atol=6e-4)       # shipped file = recomputed
assert json.loads((PRE / "lodo_reference.json").read_text())["in_distribution_random_80_20"]["n_test"] == te.sum()
report(start)

       study   n  age_sd     R2  MedAE
    GSE15745 151  22.508  0.784  5.815
    GSE42861 335  11.461  0.676  4.359
    GSE41037 394  17.712  0.628  7.798
E-GEOD-36194 433  27.522  0.609 10.847
    GSE34035 118  10.489  0.372  5.916
E-GEOD-40279 393  15.081  0.216  9.268
   TCGA_KIRC 356  12.114  0.208  6.979
E-GEOD-64511 222  25.964  0.086 16.635
   TCGA_BRCA 113  14.628 -0.161  7.313
E-GEOD-53740 193  10.275 -0.495 10.332 

                                  quantity  recomputed  manuscript match
                 in-distribution 80/20: R2     0.87731       0.877   yes
              in-distribution 80/20: MedAE     4.64520       4.650   yes
in-distribution 80/20: evaluation profiles  2142.00000    2142.000   yes
                          eligible studies    10.00000      10.000   yes
                  median study-held-out R2     0.29396       0.294   yes
               median study-held-out MedAE     7.55530       7.560   yes
                  studies with negative R2     2.00000    

## 4. Robustness to CpG subsampling and fine-tuning (Supplementary Figs. S2 and S3)

**S2.** `two_view_v1n.npy` holds the CLS of one random 50% CpG view of 2,000 downstream profiles
(rows `np.random.default_rng(0).choice(10988, 2000)`). A ridge probe fitted on complete-profile
embeddings of training profiles is applied without adaptation to the complete and subset
embeddings of the 407 test profiles among them; paired bootstrap over 5,000 resamples (seed 0).
**S3.** Fine-tuned fold-4 CLS of the same 10,988 profiles: probes as in section 3 and the mean
absolute age difference to the ten nearest neighbours. The fine-tuned classification probes are fitted
on the stored float32 embeddings, as the original run did; in float64 the study probe gives 0.950
instead of 0.949 (the logistic-regression solver stops at a slightly different point).

In [12]:
start = len(checks)
v1 = np.load(PRE / "two_view_v1n.npy").astype(np.float64)
idx = np.random.default_rng(0).choice(len(cls), len(v1), replace=False)
cos_pair = (unit[idx] * v1).sum(1)
cos_shuf = (unit[idx][np.random.default_rng(1).permutation(len(idx))] * v1).sum(1)
check("cosine, complete profile vs own 50% subset", cos_pair.mean(), 0.991, 3)
check("cosine, shuffled pairing", cos_shuf.mean(), 0.496, 3)
fit_tr = (split == "train") & np.isfinite(age)
probe = Ridge(alpha=1.0).fit(unit[fit_tr], age[fit_tr])
where = {int(g_): p_ for p_, g_ in enumerate(idx)}
ev_g = [g_ for g_ in np.where((split == "test") & np.isfinite(age))[0] if int(g_) in where]
y_, pf, pv = age[ev_g], probe.predict(unit[ev_g]), probe.predict(v1[[where[int(g_)] for g_ in ev_g]])
check("held-out profiles with a subset embedding", len(ev_g), 407, 0)
check("age probe R2, complete profiles", r2_score(y_, pf), 0.863, 3)
check("age probe R2, 50% subsets", r2_score(y_, pv), 0.835, 3)
rng = np.random.default_rng(0)
dr = [(lambda s_: r2_score(y_[s_], pf[s_]) - r2_score(y_[s_], pv[s_]))(rng.integers(0, len(y_), len(y_))) for _ in range(5000)]
check("paired difference", r2_score(y_, pf) - r2_score(y_, pv), 0.028, 3)
check("  95% CI low", np.percentile(dr, 2.5), 0.014, 3)
check("  95% CI high", np.percentile(dr, 97.5), 0.044, 3)

ft = np.load(FT / "embeddings_cls.npy").astype(np.float64)
check("fine-tuned: age probe R2", ridge(ft, fit, ev)[0], 0.938, 3)
cls_frozen = cls
cls = np.load(FT / "embeddings_cls.npy")    # classify() reads the global `cls`; float32 as stored, as in the
                                           # original run (extract_finetune_cls.py); float64 moves study to 0.950
for column, paper in [("tissue_type", 0.930), ("dataset", 0.949)]:
    check(f"fine-tuned: {column} balanced accuracy", classify(column)[0], paper, 3)
cls = cls_frozen
def age_gap(X):
    u = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    nb = NearestNeighbors(n_neighbors=11).fit(u).kneighbors(u)[1][:, 1:]
    return np.abs(age[nb] - age[:, None]).mean()
check("10-NN age gap, pretrained (years)", age_gap(cls), 8.9, 1)
check("10-NN age gap, fine-tuned (years)", age_gap(ft), 5.8, 1)
check("  shuffled pairing (years)", np.abs(age[np.random.default_rng(0).permutation(len(age))] - age).mean(), 31.4, 1)
report(start)

/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


                                  quantity  recomputed  manuscript match
cosine, complete profile vs own 50% subset     0.99058       0.991   yes
                  cosine, shuffled pairing     0.49550       0.496   yes
 held-out profiles with a subset embedding   407.00000     407.000   yes
           age probe R2, complete profiles     0.86289       0.863   yes
                 age probe R2, 50% subsets     0.83484       0.835   yes
                         paired difference     0.02805       0.028   yes
                                95% CI low     0.01445       0.014   yes
                               95% CI high     0.04365       0.044   yes
                  fine-tuned: age probe R2     0.93788       0.938   yes
 fine-tuned: tissue_type balanced accuracy     0.92995       0.930   yes
     fine-tuned: dataset balanced accuracy     0.94902       0.949   yes
         10-NN age gap, pretrained (years)     8.85400       8.900   yes
         10-NN age gap, fine-tuned (years)     5.83

## 5. Data structure (Methods "Datasets", Discussion)

Counts of the identifier-level audit (`scripts/data/data_structure_audit.py`, `results/data_audit/audit.json`)
and of the study-level quality check of the downstream corpus
(`scripts/data/downstream_integrity_check.py`). The audit reads identifiers and annotations only; the
quality check correlates each downstream profile with the copy of the same accession in the
pretraining corpus. The downstream counts are recomputed from the per-profile table.

In [13]:
start = len(checks)
au = json.loads((AUD / "audit.json").read_text())
p_ = au["pretraining"]
check("pretraining profiles", p_["n_profiles"], 169120, 0)
for part, paper in [("train", 135296), ("valid", 16912), ("heldout", 16912)]:
    check(f"  {part} partition", p_["partition"][part], paper, 0)
check("profiles with tissue annotation", p_["tissue"]["profiles_annotated"], 129373, 0)
check("  tissue categories", p_["tissue"]["categories"], 32, 0)
check("profiles with study annotation", p_["study"]["profiles_annotated"], 136860, 0)
check("  studies", p_["study"]["studies"], 1816, 0)
check("held-out profiles with study annotation", p_["study"]["heldout_profiles_annotated"], 13699, 0)
check("  whose study is in training", p_["study"]["heldout_profiles_whose_study_is_in_train"], 13695, 0)
rel = p_["individual"]["reliable_identifiers"]
check("held-out profiles with a reliable individual identifier", rel["heldout_profiles_with_identifier"], 3545, 0)
check("  sharing the individual with training", rel["heldout_profiles_whose_individual_is_also_in_train"], 1250, 0)

check("downstream profiles", len(ex), 10988, 0)
check("benchmark profiles", len(bench), 10589, 0)
dev, tst = bench[bench["bench"] == "development"], bench[bench["bench"] == "test"]
check("development pool", len(dev), 8440, 0)
check("  from the original training partition", (dev["split"] == "train").sum(), 7177, 0)
check("  from the original validation partition", (dev["split"] == "valid").sum(), 1263, 0)
check("test set", len(tst), 2149, 0)
check("benchmark profiles with an individual identifier", person.notna().sum(), 1727, 0)
check("test studies", tst["study"].nunique(), 77, 0)
check("  also in development", tst["study"].isin(set(dev["study"])).sum() == len(tst), True, 0)
check("test profiles whose tissue label is in development", tst["tissue_type"].isin(set(dev["tissue_type"])).sum(), 2146, 0)
check("  missing label", sorted(set(tst["tissue_type"]) - set(dev["tissue_type"])) == ["vaginal swab"], True, 0)
for part, paper in [("train", 1284), ("valid", 186), ("heldout", 134)]:
    check(f"test profiles in the pretraining {part} partition (same accession)", (tst["pretrain_split"] == part).sum(), paper, 0)
check("test profiles with no identifier match", tst["pretrain_split"].isna().sum(), 545, 0)
check("TCGA: participant's normal tissue in pretraining training", tst["exposure"].str.startswith("2").sum(), 174, 0)
check("same individual, other sample, in pretraining training", tst["exposure"].str.startswith("3").sum(), 65, 0)

qc = pd.read_csv(AUD / "downstream_integrity_by_study.csv").set_index("study").loc["E-GEOD-56105"]
check("GSE56105 profiles", qc["n"], 282, 0)
check("GSE56105: median r with the corpus copy", qc["r_copy_median"], 0.002, 3)
check("GSE56105 corpus copies: median r with the reference profile", qc["corpus_copy_r_ref_median"], 0.985, 3)
fs = json.loads((AUD / "frozen_probe_sensitivity.json").read_text())
r_, q_ = fs["as_reported"], fs["no_56105"]
check("excluding GSE56105: change in age R2", abs(q_["age_probe"]["r2"] - r_["age_probe"]["r2"]), 0.001, 3)
check("  largest change in balanced accuracy <= 0.003",
      max(abs(q_[k]["balanced_acc"] - r_[k]["balanced_acc"]) for k in ("tissue_probe", "dataset_probe")) <= 0.003 + 1e-9, True, 0)
report(start)

                                                           quantity   recomputed manuscript match
                                               pretraining profiles 169120.00000     169120   yes
                                                    train partition 135296.00000     135296   yes
                                                    valid partition  16912.00000      16912   yes
                                                  heldout partition  16912.00000      16912   yes
                                    profiles with tissue annotation 129373.00000     129373   yes
                                                  tissue categories     32.00000         32   yes
                                     profiles with study annotation 136860.00000     136860   yes
                                                            studies   1816.00000       1816   yes
                            held-out profiles with study annotation  13699.00000      13699   yes
                    

## 6. Genomic-rank pilot (Supplementary Note 1)

Mean cosine similarity of contextual CpG representations for genomically adjacent CpGs and for random
pairs, in the two pilot models (`results/rope_pilot/`), and the genomic locality of the production
model's static token embeddings (`results/pretrained_cls/cpg_embedding_analysis.json`).

In [14]:
start = len(checks)
rp = json.loads((RES / "rope_pilot/rope_ablation_summary.json").read_text())
A, B = rp["run_A_genomic"], rp["run_B_nogenomic"]
check("genomic positions: adjacent CpGs", A["adjacent_cos_dist1"], 0.9831, 4)
check("genomic positions: random pairs", A["random_pair_baseline"], 0.8857, 4)
check("  locality gap", A["adjacent_cos_dist1"] - A["random_pair_baseline"], 0.0974, 4)
check("sequential positions: adjacent CpGs", B["adjacent_cos_dist1"], 0.9931, 4)
check("sequential positions: random pairs", B["random_pair_baseline"], 0.9621, 4)
check("  locality gap", B["adjacent_cos_dist1"] - B["random_pair_baseline"], 0.0310, 4)
check("ratio of the gaps", (A["adjacent_cos_dist1"] - A["random_pair_baseline"]) / (B["adjacent_cos_dist1"] - B["random_pair_baseline"]), 3.14, 2)
curve = pd.read_csv(RES / "rope_pilot/rope_ablation_curve_A.csv")
assert np.isclose(curve.loc[curve["distance"] == 1, "cos"].item(), A["adjacent_cos_dist1"])
ce = json.loads((PRE / "cpg_embedding_analysis.json").read_text())
check("static embeddings: no positive locality gap", ce["adjacent_vs_random_gap"] <= 0, True, 0)
report(start)

                                   quantity  recomputed manuscript match
           genomic positions: adjacent CpGs    0.983108     0.9831   yes
            genomic positions: random pairs    0.885684     0.8857   yes
                               locality gap    0.097424     0.0974   yes
        sequential positions: adjacent CpGs    0.993115     0.9931   yes
         sequential positions: random pairs    0.962104     0.9621   yes
                               locality gap    0.031011      0.031   yes
                          ratio of the gaps    3.141600       3.14   yes
static embeddings: no positive locality gap    1.000000       True   yes


## 7. CNS tumour classification (Figs. 5 and 6, Supplementary Figs. S5–S8, Supplementary Table S3)

Frozen 256-dimensional CLS embeddings of MethylLlama and MethylGPT, and two raw-β baselines, classified with the
same linear classifier (5 × 5 cross-validation on the 2,798 Capper reference profiles; `scripts/cns/`).

- `cns/full/comparison.json`: balanced accuracy on complete profiles, every representation (Supplementary
  Fig. S6, the five raw-β variants of Methods).
- `cns/figure_data/fig5d_neighbours.csv`: neighbourhood measures of Fig. 5 per model, share of CpGs and mask seed.
- `cns/sparse_correctness.npz`: for every cell of the incomplete-input grids, each reference profile's mean
  correctness over the repeats. From it, `scripts/cns/tests_from_correctness.py` recomputes the balanced
  accuracies of Fig. 6 and the paired, Holm-adjusted tests of Supplementary Table S3 with the bootstrap seeds of
  the original run, so the intervals match.
- `cns/external/metrics.csv`: the external cohort GSE215240 (Supplementary Fig. S8).


In [15]:
start = len(checks)
import sys
sys.path.insert(0, str(ROOT / "scripts/cns"))
from tests_from_correctness import tests

CNS = RES / "cns"
full = json.loads((CNS / "full/comparison.json").read_text())["per_representation"]
check("complete profiles: MethylLlama", full["llama_cls"]["full_BA"], 0.95, 2)
check("complete profiles: MethylGPT", full["gpt_cls"]["full_BA"], 0.84, 2)
for rep, v in (("raw_top5k", 0.972), ("raw_top10k", 0.971), ("raw_pca256", 0.964), ("raw_top32k", 0.963), ("raw_all", 0.954)):
    check(f"complete profiles: {rep}", full[rep]["full_BA"], v, 3)

nb5 = pd.read_csv(CNS / "figure_data/fig5d_neighbours.csv")
m = nb5.groupby(["model", "level"])[["category_majority", "class_knn"]].mean()
check("Fig. 5a: neighbours of own class, MethylLlama", m.loc[("llama", 100.0), "class_knn"], 0.84, 2)
check("Fig. 5a: neighbours of own class, MethylGPT", m.loc[("gpt", 100.0), "class_knn"], 0.57, 2)
check("5% of CpGs: category kept, MethylLlama", m.loc[("llama", 5.0), "category_majority"], 0.95, 2)
check("5% of CpGs: category kept, MethylGPT", m.loc[("gpt", 5.0), "category_majority"], 0.03, 2)

A, T = tests(np.load(CNS / "sparse_correctness.npz"))
ba = A.set_index(["scenario", "level", "rep"])["ba"]
check("5%, trained and tested at 5%: MethylLlama", ba[("B", 5.0, "llama_cls")], 0.83, 2)
check("5%, trained and tested at 5%: best raw (PCA-256)", ba[("B", 5.0, "raw_pca256")], 0.56, 2)
check("5%, trained and tested at 5%: MethylGPT", ba[("B", 5.0, "gpt_cls")], 0.49, 2)
check("Fig. 6c: MethylLlama, classifier trained on complete profiles", ba[("A", 5.0, "llama_cls")], 0.83, 2)
check("Fig. 6c: raw top-5,000, classifier trained on complete profiles", ba[("A", 5.0, "raw_top5k")], 0.93, 2)
check("Fig. 6c: raw top-5,000, classifier trained at 5%", ba[("B", 5.0, "raw_top5k")], 0.43, 2)

# Supplementary Table S3: difference, 95% CI and Holm-adjusted p (MethylLlama minus comparator)
S3 = {  # level: (MethylGPT row, best raw row), each (diff, ci_lo, ci_hi, p_holm) as printed
    0.5: ((0.055, 0.047, 0.064, 0.006), (0.098, 0.091, 0.106, 0.006)),
    1: ((0.172, 0.161, 0.184, 0.006), (0.200, 0.190, 0.211, 0.006)),
    2: ((0.307, 0.293, 0.322, 0.006), (0.290, 0.278, 0.302, 0.006)),
    5: ((0.342, 0.328, 0.357, 0.006), (0.265, 0.254, 0.277, 0.006)),
    10: ((0.285, 0.270, 0.300, 0.006), (0.164, 0.152, 0.175, 0.006)),
    20: ((0.224, 0.210, 0.239, 0.006), (0.071, 0.060, 0.082, 0.006)),
    50: ((0.144, 0.129, 0.158, 0.006), (0.013, 0.004, 0.024, 0.044)),
    60: ((0.130, 0.116, 0.145, 0.006), (0.001, -0.008, 0.011, 0.728)),
    70: ((0.123, 0.109, 0.138, 0.006), (-0.008, -0.018, 0.002, 0.208)),
    80: ((0.121, 0.107, 0.135, 0.006), (-0.013, -0.022, -0.003, 0.044)),
    90: ((0.113, 0.099, 0.128, 0.006), (-0.018, -0.027, -0.009, 0.006)),
    100: ((0.111, 0.096, 0.128, 0.006), (-0.019, -0.028, -0.010, 0.006)),
}
tb = T[T.scenario == "B"].set_index(["level", "vs"])
for lv, rows in S3.items():
    for vs, vals in zip(("MethylGPT", "best raw"), rows):
        r = tb.loc[(float(lv), vs)]
        for col, v in zip(("diff", "ci_lo", "ci_hi", "p_holm"), vals):
            check(f"Table S3 {lv:g}% vs {vs}: {col}", r[col], v, 3)
sig = tb.xs("best raw", level="vs")
check("MethylLlama significantly above best raw at every level 0.5-50%",
      bool(((sig["diff"] > 0) & sig.significant).loc[[0.5, 1, 2, 5, 10, 20, 50]].all()), True, 0)
check("no significant difference at 60% and 70%", bool((~sig.significant).loc[[60, 70]].all()), True, 0)
check("raw beta significantly above from 80%", bool(((sig["diff"] < 0) & sig.significant).loc[[80, 90, 100]].all()), True, 0)
check("MethylLlama above MethylGPT at every level (Holm p < 0.05)",
      bool(((tb.xs("MethylGPT", level="vs")["diff"] > 0) & tb.xs("MethylGPT", level="vs").significant).all()), True, 0)

ext = pd.read_csv(CNS / "external/metrics.csv").set_index("rep")["ba"]
check("external cohort: MethylLlama", ext["llama_cls"], 0.80, 2)
check("external cohort: MethylGPT", ext["gpt_cls"], 0.59, 2)
report(start)


                                                       quantity  recomputed manuscript match
                                 complete profiles: MethylLlama     0.95150       0.95   yes
                                   complete profiles: MethylGPT     0.84130       0.84   yes
                                   complete profiles: raw_top5k     0.97165      0.972   yes
                                  complete profiles: raw_top10k     0.97128      0.971   yes
                                  complete profiles: raw_pca256     0.96392      0.964   yes
                                  complete profiles: raw_top32k     0.96310      0.963   yes
                                     complete profiles: raw_all     0.95447      0.954   yes
                  Fig. 5a: neighbours of own class, MethylLlama     0.84150       0.84   yes
                    Fig. 5a: neighbours of own class, MethylGPT     0.57490       0.57   yes
                         5% of CpGs: category kept, MethylLlama     0.

## 8. Contrastive-weight pilot (Supplementary Note 2)

Four arms pretrained on a 9,000-profile subset with InfoNCE weights 0, 0.01, 0.05 and 0.2 (`objective_pilot/`),
each evaluated with the manuscript's own pipelines. `objective_pilot/pilot_table.csv` is what
`objective_pilot/summarize_pilot.py` writes from the per-arm files in `objective_pilot/arms/`.


In [16]:
start = len(checks)
PIL = RES / "objective_pilot"
pt = pd.read_csv(PIL / "pilot_table.csv").set_index("arm")
a0, a5 = pt.loc["lam0.0_seed42_ep19"], pt.loc["lam0.05_seed42_ep19"]
check("disjoint views, same study and tissue: lambda 0 (%)", 100 * a0["hn_study_tissue_top1"], 28.5, 1)
check("disjoint views, same study and tissue: lambda 0.05 (%)", 100 * a5["hn_study_tissue_top1"], 55.1, 1)
arms = ["lam0.0_seed42_ep19", "lam0.01_seed42_ep19", "lam0.05_seed42_ep19", "lam0.2_seed42_ep19"]
check("monotonic increase over the four weights", bool(np.all(np.diff(pt.loc[arms, "hn_study_tissue_top1"]) > 0)), True, 0)
check("disjoint views, all 16,912 held-out: lambda 0 (%)", 100 * a0["disjoint_top1"], 20.0, 1)
check("disjoint views, all 16,912 held-out: lambda 0.05 (%)", 100 * a5["disjoint_top1"], 43.7, 1)
check("reconstruction Pearson r: lambda 0", a0["recon_pcc"], 0.930, 3)
check("reconstruction Pearson r: lambda 0.05", a5["recon_pcc"], 0.931, 3)
lo = pt.loc["lam0.2_seed42_ep19"]
check("lambda 0.2 below lambda 0 in frozen age, tissue and study probes",
      bool((lo[["age_ridge_r2", "tissue_type_bal_acc", "dataset_bal_acc"]] < a0[["age_ridge_r2", "tissue_type_bal_acc", "dataset_bal_acc"]]).all()), True, 0)
report(start)


                                                        quantity  recomputed manuscript match
             disjoint views, same study and tissue: lambda 0 (%)    28.52200       28.5   yes
          disjoint views, same study and tissue: lambda 0.05 (%)    55.11200       55.1   yes
                        monotonic increase over the four weights     1.00000       True   yes
               disjoint views, all 16,912 held-out: lambda 0 (%)    19.98000       20.0   yes
            disjoint views, all 16,912 held-out: lambda 0.05 (%)    43.66700       43.7   yes
                              reconstruction Pearson r: lambda 0     0.93001       0.93   yes
                           reconstruction Pearson r: lambda 0.05     0.93100      0.931   yes
lambda 0.2 below lambda 0 in frozen age, tissue and study probes     1.00000       True   yes


## Summary

In [17]:
table = pd.DataFrame(checks)
print(f"{(table['match'] == 'yes').sum()} of {len(table)} manuscript values reproduced")
mismatch = table[table["match"] != "yes"]
if len(mismatch):
    print(mismatch.to_string(index=False))

403 of 403 manuscript values reproduced
